In [ ]:
# RQ3a: is the offline phase worth paying for 1 to 10 000 queries? (spec 2, H3a; decision 28 runs the
# session check at 128 MiB x 1 KiB only)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

SESSION = repo / "results/raw/p6-20261113/session"
session = frame.load_session(SESSION, platform)
t = plots.build(df, status, session=session)
cmp_meta = json.loads((SESSION / "comparison.json").read_text())
print(f"grid run {cmp_meta['grid_run']}, session run {cmp_meta['session_run']}, Q = {cmp_meta['queries']},",
      f"{session[frame.KEY].drop_duplicates().shape[0]} cells")

In [ ]:
# Session check (spec 5): per cell one cold offline phase, then Q = 100 consecutive online queries in one process,
# against the grid: computed = median(offline) + Q x median(online). Spec 5 asks for an explanation of every total
# more than 5 % off at Q = 100. Then the relative difference in % at Q = 1, 10 and 100 per arm and quantity.
print(session_check.report(session, cmp_meta["queries"]))
rel = session.pivot_table(index=["arm", "quantity"], columns="q", values="rel_diff")
print((100 * rel).round(1).to_string())

In [ ]:
# H3a: break-even Q from the grid medians (amortise.break_even_Q, closed form): the smallest Q at which arm a's
# amortised cost per query, offline / Q + online, is at most arm b's. None: never. 1: from the first query.
# Price pairs first, then single-server against two-server. 1 KiB records at every size both arms ran.
PAIRS = [("S1", "S0-cpp"), ("S2", "S0-cpp"), ("M1", "M0"), ("S1", "M1"), ("S2", "M1"), ("M1", "S1"), ("M1", "S2")]
med = t.medians
rows = []
for a, b in PAIRS:
    for db in sorted(med.db_bytes.unique()):
        ca, cb = plots._cell(med, a, db, 1 << 10), plots._cell(med, b, db, 1 << 10)
        if ca is not None and cb is not None:
            rows.append({"a": a, "b": b, "db": fmt_size(int(db)), **{k: amortise.break_even_Q(ca, cb, k)
                                                                      for k in ("bytes", "client_s", "server_s", "total_s")}})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Which arm costs least per query at each Q, at 128 MiB x 1 KiB (S3 is per retrieved bit, so it is listed apart):
# amortised bytes and client + server seconds per query from the grid medians.
am = t.amortised[(t.amortised.db_bytes == CELL[0]) & (t.amortised.record_bytes == CELL[1])]
for kind in ("bytes", "total_s"):
    tab = am.pivot_table(index="arm", columns="Q", values=kind)
    print(f"{kind} per query\n{tab.to_string(float_format=lambda v: f'{v:.4g}')}")
    ver = tab.loc[[a for a in ("S1", "S2", "M1") if a in tab.index]]
    print("cheapest arm:", {q: tab[q].idxmin() for q in tab.columns})
    print("cheapest verifiable arm:", {q: ver[q].idxmin() for q in ver.columns}, "\n")
s3 = t.amortised[(t.amortised.arm == "S3") & (t.amortised.db_bytes == CELL[0])]
print("S3 at 128 MiB, one bit per query:\n", s3[["Q", "bytes", "total_s"]].to_string(index=False))

In [ ]:
# F4 (amortised cost against Q at the largest size where S1, S2 and M1 all ran) and F8 (measured against
# computed totals for Q = 1, 10, 100 and 1 000).
for name in ("F4_amortised", "F8_session"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F[48]_*")))